# Root / CSC cluster and trajectory — Patient 1 (03H096 / PB2)

**What this notebook does (in order)**

Step 1 (MultiVI) only gave a **starting** Leiden partition (resolution 0.8). That is not the published root. Here we:

1. Load the cleaned MultiVI object
2. Count accessible ATAC peaks per cell
3. Compare clusters with ANOVA + Tukey HSD — the cluster with the **highest global chromatin accessibility** is the candidate root / CSC population (cluster **6** in Patient 1)
4. Run **PAGA** on that partition and **diffusion pseudotime (DPT)** from a cell inside the high-accessibility cluster, toward terminal states
5. Ask whether the same cluster is also more open at **enhancers**, **promoters**, and repeat families (LTR / DNA / LINE / SINE)
6. Write the `*_Trajectory.h5ad` object used in the figures

**Objects**
- **Reads:** `DATA_DIR / "01_Trimodal_integration_MultiVI/cleaned_MultiVI/Teaseq_Multi_VI_PB2_Cleaned.h5ad"`
- **Reads (peak classes):** `DATA_DIR / "02_Trajectory_analysis/peak_annotation/teapb2_peaks_annotated.tsv"`
- **Creates:** `DATA_DIR / "02_Trajectory_analysis/trajectory_objects/MultiVI_Patient1_Relapse_03h096_Trimodal_Cleaned_Trajectory.h5ad"`

The same procedure was run for every patient; only the root cluster / root-cell index change.


## Paths and settings


In [ ]:
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import muon
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse
from scipy.stats import f_oneway
from scipy.stats import f as fdist
from sklearn.preprocessing import QuantileTransformer
from statsmodels.stats.multicomp import pairwise_tukeyhsd

DATA_DIR = Path("PATH_TO_DATA")  # <-- set this to your local data root
OUT_DIR = DATA_DIR / "02_Trajectory_analysis/trajectory_objects"
OUT_DIR.mkdir(parents=True, exist_ok=True)


## Load the cleaned MultiVI object

`Cluster_Final` is the working cluster partition. ForceAtlas coordinates are for visualization only.


In [ ]:
adata = ad.read_h5ad(
    DATA_DIR
    / "01_Trimodal_integration_MultiVI/cleaned_MultiVI"
    / "Teaseq_Multi_VI_PB2_Cleaned.h5ad"
)
adata


In [ ]:
muon.pl.embedding(
    adata,
    basis="X_draw_graph_fa",
    color=["Cluster_Final"],
    frameon=False,
    ncols=1,
    size=100,
)


## Accessible peaks per cell

Methods: the number of ATAC peaks per cell with at least one read. The cluster with the highest mean is the candidate root / CSC. This is a manual check, not an automatic label from Leiden.


In [ ]:
feature_types = adata.var["feature_types"]
peak_mask = feature_types == "Peaks"
gene_mask = feature_types == "Gene Expression"

X_peaks = adata[:, peak_mask].X
X_genes = adata[:, gene_mask].X
if sparse.issparse(X_peaks):
    adata.obs["n_peaks"] = np.asarray((X_peaks > 0).sum(axis=1)).ravel()
    adata.obs["n_genes"] = np.asarray((X_genes > 0).sum(axis=1)).ravel()
else:
    adata.obs["n_peaks"] = (X_peaks > 0).sum(axis=1)
    adata.obs["n_genes"] = (X_genes > 0).sum(axis=1)

adata.obs[["n_peaks", "n_genes"]].describe()


## Compare chromatin opening across clusters (ANOVA)

One-way ANOVA on peaks/cell across `Cluster_Final`, then Tukey HSD for pairwise tests. If a cluster stands out with higher accessibility, that is the root used below. For Patient 1 this is cluster **6**.


In [ ]:
def _format_p_from_logsf(logp):
    """Readable p-value string from natural-log(p)."""
    if not np.isfinite(logp):
        return "p=NA", None
    neglog10p = (-logp) / np.log(10)
    if neglog10p > 300:
        return f"p < 1e-{int(np.floor(neglog10p))}", neglog10p
    return f"p={np.exp(logp):.2e}", neglog10p


def plot_cluster_openness_with_stats(adata, metric, highlight_max=True):
    """Mean ± 95% CI of `metric` per Cluster_Final, with ANOVA + Tukey HSD."""
    if "Cluster_Final" not in adata.obs:
        raise KeyError("`Cluster_Final` is missing from adata.obs")
    if metric not in adata.obs:
        raise KeyError(f"`{metric}` is missing from adata.obs")

    cl = adata.obs["Cluster_Final"]
    if not pd.api.types.is_categorical_dtype(cl):
        adata.obs["Cluster_Final"] = cl.astype("category")
        cl = adata.obs["Cluster_Final"]
    categories = list(cl.cat.categories)

    raw_colors = adata.uns.get("Cluster_Final_colors", None)
    if raw_colors is None:
        raise ValueError("adata.uns['Cluster_Final_colors'] is missing")
    colors = list(raw_colors)
    if len(colors) > len(categories):
        colors = colors[: len(categories)]
    if len(colors) < len(categories):
        raise ValueError("Not enough colors in Cluster_Final_colors for the categories")
    color_map = dict(zip(categories, colors))

    df = adata.obs[["Cluster_Final", metric]].copy()
    df[metric] = pd.to_numeric(df[metric], errors="coerce")
    df = df.dropna(subset=["Cluster_Final", metric])
    df["Cluster_Final"] = df["Cluster_Final"].astype("category")
    df["Cluster_Final"] = df["Cluster_Final"].cat.set_categories(categories, ordered=True)

    means = df.groupby("Cluster_Final", observed=True)[metric].mean().reindex(categories)
    counts = df.groupby("Cluster_Final", observed=True)[metric].count().reindex(categories)
    stds = df.groupby("Cluster_Final", observed=True)[metric].std().reindex(categories)
    sem = stds / np.sqrt(counts)
    ci95 = (1.96 * sem).fillna(0.0)

    groups = [df.loc[df["Cluster_Final"] == c, metric].values for c in categories]
    groups_anova = [g for g in groups if len(g) >= 2]
    k = len(groups_anova)
    N = sum(len(g) for g in groups_anova)
    dfn, dfd = k - 1, N - k

    if k >= 2:
        F, p = f_oneway(*groups_anova)
        logp = fdist.logsf(F, dfn, dfd)
        p_text, neglog10p = _format_p_from_logsf(logp)
        anova_title = f"ANOVA F({dfn},{dfd})={F:.2f}, {p_text}"
    else:
        F = np.nan
        p_text, neglog10p = "p=NA", None
        anova_title = "ANOVA not applicable (need ≥2 groups with n≥2)"

    tukey = None
    try:
        if df["Cluster_Final"].nunique() >= 2:
            tukey = pairwise_tukeyhsd(df[metric].values, df["Cluster_Final"].astype(str).values)
    except Exception:
        tukey = None

    x = np.arange(len(categories))
    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.bar(
        x,
        means.values,
        yerr=ci95.values,
        color=[color_map[cat] for cat in categories],
        edgecolor="black",
        linewidth=1.2,
        alpha=0.9,
        capsize=5,
        error_kw=dict(lw=1.2, capsize=5, capthick=1.2),
    )
    if highlight_max and np.isfinite(means.values).any():
        max_idx = int(np.nanargmax(means.values))
        bars[max_idx].set_hatch("//")
        bars[max_idx].set_linewidth(2.0)
        print(f"Highest mean {metric}: cluster {categories[max_idx]}")

    ax.set_xticks(x)
    ax.set_xticklabels([str(c) for c in categories])
    ax.set_xlabel("Cluster_Final")
    if metric == "n_peaks":
        ax.set_ylabel("Mean accessible ATAC peaks per cell")
        ax.set_title(f"Accessible ATAC peaks per cell by cluster\n({anova_title})")
    else:
        ax.set_ylabel(f"Average {metric}")
        ax.set_title(f"Average {metric} per cluster\n({anova_title})")
    ax.grid(axis="y", linestyle="--", alpha=0.4)
    plt.tight_layout()
    plt.show()

    print(f"\n[Summary] {metric}")
    print(pd.DataFrame({
        "Mean": means, "Count": counts, "Std": stds, "SEM": sem,
        "CI95_Lower": means - ci95, "CI95_Upper": means + ci95,
    }))
    if np.isfinite(F):
        print(f"\n[ANOVA] F({dfn},{dfd}) = {F:.6g}  {p_text}")
        if neglog10p is not None:
            print(f"[ANOVA] -log10(p) ≈ {neglog10p:.2f}")
    if tukey is not None:
        print("\n[Tukey HSD]")
        print(pd.DataFrame(tukey._results_table.data[1:], columns=tukey._results_table.data[0]).to_string(index=False))


plot_cluster_openness_with_stats(adata, "n_peaks")


## PAGA and diffusion pseudotime from the root

The root is not chosen by Leiden. It is the high-accessibility cluster from the ANOVA above. For Patient 1:

- `root_cell = 6` — that cluster
- `root_index = 2` — which cell inside the cluster is used as `iroot` for DPT

PAGA draws the cluster graph. DPT then orders cells from that root toward terminal states (T1 / T2 in Patient 1). ForceAtlas is only the layout.


In [ ]:
# Patient 1 (PB2 / 03H096)
root_cell = 6
root_index = 2

sc.tl.paga(adata, groups="Cluster_Final")

cluster_names = adata.obs["Cluster_Final"].cat.categories
cluster_centers = np.array([
    adata.obsm["X_draw_graph_fa"][adata.obs["Cluster_Final"] == cl].mean(axis=0)
    for cl in cluster_names
])
adata.uns["paga"]["pos"] = cluster_centers

sc.pl.paga(
    adata,
    color="Cluster_Final",
    pos=adata.uns["paga"]["pos"],
    threshold=0.3,
    edge_width_scale=0.3,
)

root_indices = np.flatnonzero(adata.obs["Cluster_Final"] == str(root_cell))
if len(root_indices) > root_index:
    adata.uns["iroot"] = root_indices[root_index]
else:
    print(f"Warning: index {root_index} out of range; using the first cell in cluster {root_cell}.")
    adata.uns["iroot"] = root_indices[0]

sc.tl.dpt(adata)

pseudotime = adata.obs["dpt_pseudotime"].values.reshape(-1, 1)
transformer = QuantileTransformer(n_quantiles=10, random_state=0)
adata.obs["dpt_pseudotime"] = transformer.fit_transform(pseudotime).flatten()

sc.pl.draw_graph(
    adata,
    size=400,
    color="dpt_pseudotime",
    legend_fontsize=20,
    color_map="jet",
)

muon.pl.embedding(
    adata,
    basis="X_draw_graph_fa",
    color=["Cluster_Final", "dpt_pseudotime"],
    frameon=False,
    ncols=2,
    size=100,
)


## Peak classes: enhancers, promoters, repeats

Global opening (all peaks) identified the root. We next split peaks by annotation and repeat the same ANOVA.

The annotation table is the per-peak `regulatory_element_type` / `repClass` file for this sample.


In [ ]:
annot = pd.read_csv(
    DATA_DIR / "02_Trajectory_analysis/peak_annotation/teapb2_peaks_annotated.tsv",
    sep="\t",
)

annot_collapsed = (
    annot.groupby(["seqnames", "start", "end"])["regulatory_element_type"]
    .apply(lambda s: ";".join(sorted(set(s.dropna().astype(str)))) if s.dropna().size else np.nan)
    .reset_index()
    .rename(columns={"seqnames": "chr", "regulatory_element_type": "RE"})
)
repclass_collapsed = (
    annot.groupby(["seqnames", "start", "end"])["repClass"]
    .apply(lambda s: ";".join(sorted(set(s.dropna().astype(str)))) if s.dropna().size else np.nan)
    .reset_index()
    .rename(columns={"seqnames": "chr", "repClass": "repClass_collapsed"})
)

var_df = adata.var.reset_index()
var_df["start"] = pd.to_numeric(var_df["start"], errors="coerce")
var_df["end"] = pd.to_numeric(var_df["end"], errors="coerce")
var_merged = var_df.merge(annot_collapsed, on=["chr", "start", "end"], how="left").merge(
    repclass_collapsed, on=["chr", "start", "end"], how="left"
)
adata.var["RE"] = var_merged["RE"].values
for cls in ["LTR", "DNA", "LINE", "SINE"]:
    adata.var[cls] = var_merged["repClass_collapsed"].fillna("").str.contains(cls).values

adata.var[["feature_types", "chr", "start", "end", "RE", "LTR", "DNA", "LINE", "SINE"]].head()


## Accessibility of each peak class per cluster

Subset the peak matrix, calculate total ATAC-seq counts per cell, reuse the same ANOVA plot (same cluster colors).


In [ ]:
def subset_and_count_peaks(adata, mask, name):
    sub = adata[:, mask].copy()
    X = sub.X
    if sparse.issparse(X):
        sub.obs["n_peaks"] = np.asarray(X.sum(axis=1)).ravel()
    else:
        sub.obs["n_peaks"] = np.asarray(X.sum(axis=1)).ravel()
    sub.uns["Cluster_Final_colors"] = adata.uns["Cluster_Final_colors"]
    print(name, sub.shape, "mean", float(sub.obs["n_peaks"].mean()))
    plot_cluster_openness_with_stats(sub, "n_peaks")
    return sub


print("Enhancers")
subset_and_count_peaks(adata, adata.var["RE"] == "Enhancer", "Enhancer")

print("Promoters")
subset_and_count_peaks(
    adata, adata.var["RE"].isin(["Promoter", "Promoter/Enhancer"]), "Promoter"
)

for cls in ["LTR", "DNA", "LINE", "SINE"]:
    print(cls)
    subset_and_count_peaks(adata, adata.var[cls] == True, cls)


## Save the trajectory object

This file carries `Cluster_Final`, `n_peaks`, PAGA, and `dpt_pseudotime`. Downstream notebooks (TotalVI, SCENIC+, figures) should reload it rather than re-running this step.


In [ ]:
out_path = OUT_DIR / "MultiVI_Patient1_Relapse_03h096_Trimodal_Cleaned_Trajectory.h5ad"
adata.write(out_path)
print("Wrote", out_path)
